In [1]:
# Install PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")

import torch.nn as nn

import tiktoken

PyTorch version: 2.9.1


## GPT Model

Implement the LLM model architecture used in OpenAI GPT.

The purpose of a *Normalization Layer* is to improve the numerical stability and speedup convergence by having a distribution with zero mean and unit variance, which avoids vanishing or exploding gradients. 
In GTP-2 it is applied before and after multi-head attention, before the final output layer.

In [2]:
# Set model configuration
GPT_CONFIG_124M = {
  "vocab_size": 50257,    # Vocabulary size
  "context_length": 1024, # Context length
  "emb_dim": 768,         # Embedding dimension
  "n_heads": 12,          # Number of attention heads
  "n_layers": 12,         # Number of layers
  "drop_rate": 0.1,       # Dropout rate
  "qkv_bias": False       # Query-Key-Value bias
}

In [3]:
# GPT model
class DummyGPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    # Create embedding layers
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    # Placeholder for TransformerBlock
    self.trf_blocks = nn.Sequential(
      *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    # Use a placeholder for LayerNorm
    self.final_norm = DummyLayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )
  
  # Forward pass computation
  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

class DummyTransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()

  def forward(self, x):
    return x

class DummyLayerNorm(nn.Module):
  def __init__(self, normalized_shape, eps=1e-5):
    super().__init__()

  def forward(self, x):
    return x

In [9]:
# Use GPT tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Test tokenizing with two batches
batch = []
txt_1 = "Every effort moves you"
txt_2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt_1)))
batch.append(torch.tensor(tokenizer.encode(txt_2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [10]:
# Test dummy model
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6755, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


In [ ]:
# Test normalizing an activation layer
torch.manual_seed(123)
batch_example = torch.randn(2, 5)

# Create an activation layer
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print(f"Layer outputs: {out}")
print(f"Mean: {mean}")
print(f"Var: {var}")

# Normalize the layer outputs
out_norm = (out - mean) / torch.sqrt(var)
print(f"\nNormalized layer outputs: {out_norm}")
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print(f"Mean: {mean}")
print(f"Var: {var}")

tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])
Layer outputs: tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)
Mean: tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Var: tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)

Normalized layer outputs: tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
Mean: tensor([[9.9341e-09],
        [1.9868e-08]], grad_fn=<MeanBackward1>)
Var: tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [ ]:
# Normalization layer
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()

    # Small threshold constant
    self.eps = 1e-5

    # Scale and shift trainable layer
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))

  # Forward pass computation
  def forward(self, x):
    # Compute mean and variance of input tensor
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)

    # For normalization, add a small denominator constant to prevent division by zero
    norm_x = (x - mean) / torch.sqrt(var + self.eps)

    # Scale and shift
    return self.scale * norm_x + self.shift

In [15]:
# Test layer normalization
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
print(f"Mean: {mean}")
print(f"Variance: {var}")

Mean: tensor([[-2.9802e-08],
        [ 0.0000e+00]], grad_fn=<MeanBackward1>)
Variance: tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


## GELU Activations

In [ ]:
# 